# 01 – Build the Best FPL Lineup (2025/26)

This notebook fetches current-season player data and builds leaderboards & a simple lineup suggestion.

In [ ]:
# If running first time, ensure deps are installed:
# !pip install -r ../requirements.txt

import pandas as pd
from src.fpl_data.understat import fetch_understat_players, topn

df = fetch_understat_players(2025)
df.head(3)

In [ ]:
# Leaderboards (min 270 minutes to avoid tiny samples)
display(topn(df, "xG", n=10, min_minutes=270))
display(topn(df, "xA", n=10, min_minutes=270))
display(topn(df, "shots", n=10, min_minutes=270))
display(topn(df, "key_passes", n=10, min_minutes=270))
display(topn(df, "xG_per90", n=10, min_minutes=270))
display(topn(df, "xA_per90", n=10, min_minutes=270))

## Simple lineup heuristic
Pick 1 GK, 3 DEF, 4 MID, 3 FWD (tweak as you like).

In [ ]:
# Rough slot mapping and heuristic scores
def map_slot(p):
    pos = str(p).upper()
    if "GK" in pos:
        return "GK"
    if any(k in pos for k in ["CB","LB","RB","WB","LWB","RWB"]):
        return "DEF"
    if any(k in pos for k in ["FW","CF","ST"]):
        return "FWD"
    return "MID"

df["slot"] = df["pos"].map(map_slot)

df["fwd_score"] = df["xG_per90"] + 0.7*df["xA_per90"]
df["mid_score"] = df["xG_per90"] + 0.7*df["xA_per90"]
df["def_score"] = df["key_passes_per90"]

best_gk = df[df["slot"]=="GK"].sort_values("minutes", ascending=False).head(1)[["player","team","minutes"]]
best_def = df[df["slot"]=="DEF"].sort_values("def_score", ascending=False).head(3)[["player","team","def_score","minutes"]]
best_mid = df[df["slot"]=="MID"].sort_values("mid_score", ascending=False).head(4)[["player","team","mid_score","minutes"]]
best_fwd = df[df["slot"]=="FWD"].sort_values("fwd_score", ascending=False).head(3)[["player","team","fwd_score","minutes"]]

print("Suggested XI (heuristic only, no price cap):")
display(best_gk); display(best_def); display(best_mid); display(best_fwd)